# 短视频平台创作者经营分析：资源重配、验证设计与上线门槛

分析顺序如下：

1. 先确认问题发生在哪个阶段；
2. 再确认资源是否偏离长期价值；
3. 再做两段式准入与排序校正；
4. 最后判断这个方向是否值得进入实验。


## 0. 分析目标

核心问题：**当流量和现金激励都有限时，平台是否把资源给到了更值得长期扶持的创作者？**

该问题需要同时考察：
- 供给漏斗；
- cohort 稳定性；
- 曝光与收入的集中度差异；
- 关键人群识别；
- 新规则在同规模下是否改善主指标；
- 改善是否伴随可接受的迁移成本与执行风险。


In [ ]:
import pandas as pd
import numpy as np

creator = pd.read_csv('../data/creator_profile.csv', parse_dates=['join_date'])
content = pd.read_csv('../data/content_performance.csv', parse_dates=['publish_date'])
weekly = pd.read_csv('../data/creator_weekly_activity.csv')
incentive = pd.read_csv('../data/creator_incentive.csv')

print('creator rows:', len(creator))
print('content rows:', len(content))
print('weekly rows:', len(weekly))
print('incentive rows:', len(incentive))


## 1. 先看供给漏斗

如果主要损耗发生在注册到首发之前，那么资源动作应偏向拉新与首发激活；
如果主要损耗发生在首发之后到稳定经营之间，那么策略重点就应该放在持续经营。


In [ ]:
funnel = pd.DataFrame({
    'stage': ['注册创作者','完成首发','连续活跃≥2周','连续活跃≥4周','已开通变现','稳定经营且已开通变现'],
    'creator_cnt': [
        len(creator),
        int(creator['has_first_publish'].sum()),
        int(((creator['has_first_publish'] == 1) & (creator['active_weeks'] >= 2)).sum()),
        int(((creator['has_first_publish'] == 1) & (creator['active_weeks'] >= 4)).sum()),
        int(((creator['has_first_publish'] == 1) & (creator['active_weeks'] >= 4) & (creator['monetization_opened'] == 1)).sum()),
        int(((creator['has_first_publish'] == 1) & (creator['active_weeks'] >= 4) & (creator['monetization_opened'] == 1) & (creator['stable_updater'] == 1)).sum())
    ]
})
funnel['share_of_registered'] = funnel['creator_cnt'] / funnel.loc[0, 'creator_cnt']
funnel['step_conversion'] = funnel['creator_cnt'] / funnel['creator_cnt'].shift(1)
funnel


### 业务判断

主要损耗集中在首发后的持续经营阶段。后续资源动作应优先提高已启动创作者进入稳定经营状态的概率。


## 2. Cohort 检查：避免把短观察窗误读成趋势变化


In [ ]:
cohort = (
    creator.assign(join_month=creator['join_date'].dt.to_period('M').astype(str))
           .groupby('join_month')
           .agg(
               creators=('creator_id', 'count'),
               publish_rate=('has_first_publish', 'mean'),
               retention_30d=('retention_30d_rate', 'mean'),
               monetization_open_rate=('monetization_opened', 'mean')
           )
           .reset_index()
)
cohort.tail(12)


### 业务判断

各 cohort 口径整体平稳，说明问题更可能来自资源分配逻辑而非单批样本质量突变。最新 cohort 样本较小、观察窗更短，更适合作为预警信号。


## 3. 比较曝光集中度与价值集中度

这一部分用于比较当前资源分布与长期价值贡献的匹配程度。


In [ ]:
pareto = creator[['creator_id', 'total_supported_exposure', 'total_revenue']].copy()
pareto = pareto.sort_values('total_supported_exposure', ascending=False).reset_index(drop=True)
pareto['creator_share'] = (pareto.index + 1) / len(pareto)
pareto['cum_exposure_share'] = pareto['total_supported_exposure'].cumsum() / pareto['total_supported_exposure'].sum()
pareto['cum_revenue_share'] = pareto['total_revenue'].cumsum() / pareto['total_revenue'].sum()
pareto.loc[(pareto['creator_share'] - 0.20).abs().idxmin(), ['creator_share', 'cum_exposure_share', 'cum_revenue_share']]


### 业务判断

曝光累计曲线显著高于收入累计曲线，说明资源继承性强于长期价值，这构成后续资源重配的前提。


## 4. 识别关键人群

资源占用与长期价值需要分开识别，避免单一总分掩盖错配结构。


In [ ]:
creator['segment'] = np.select([
    (creator['exposure_pct_rank'] >= 0.60) & (creator['efficiency_pct_rank'] <= 0.52),
    (creator['retention_pct_rank'] >= 0.65) & (creator['efficiency_pct_rank'] >= 0.65) & (creator['resource_pct_rank'] <= 0.68),
    (creator['retention_pct_rank'] >= 0.72) & (creator['efficiency_pct_rank'] >= 0.72),
    (creator['exposure_pct_rank'] <= 0.25) & (creator['efficiency_pct_rank'] <= 0.35)
], ['高曝光低变现', '高潜低激励', '高价值稳定供给', '低价值供给'], default='一般供给')

segment_summary = (
    creator.groupby('segment')
           .agg(
               creators=('creator_id', 'count'),
               avg_retention_30d=('retention_30d_rate', 'mean'),
               avg_rev_per_1k_exposure=('rev_per_1k_exposure', 'mean'),
               exposure=('total_supported_exposure', 'sum')
           )
           .reset_index()
)
segment_summary['creator_share'] = segment_summary['creators'] / segment_summary['creators'].sum()
segment_summary['exposure_share'] = segment_summary['exposure'] / segment_summary['exposure'].sum()
segment_summary.sort_values('creators', ascending=False)


### 业务判断

- **高曝光低变现**：适合进入复核与降配流程；
- **高潜低激励**：适合进入快速晋升池；
- **高价值稳定供给**：适合做稳态扶持基线池。


## 5. 两段式策略设计

首轮策略保留旧规则主体，先通过准入过滤控制明显不适合继续强扶持的人群，再补入长期价值信号。


In [ ]:
eligible = creator[creator['has_first_publish'] == 1].copy()

# 对效率指标缩尾，避免极端值直接主导排序
low = eligible['rev_per_1k_exposure'].quantile(0.01)
high = eligible['rev_per_1k_exposure'].quantile(0.99)
eligible['efficiency_winsorized'] = eligible['rev_per_1k_exposure'].clip(low, high)

# 分位化处理
eligible['old_rule_pct'] = eligible['old_rule_score'].rank(pct=True)
eligible['retention_pct'] = eligible['retention_30d_rate'].rank(pct=True)
eligible['efficiency_pct'] = eligible['efficiency_winsorized'].rank(pct=True)

# 准入过滤：示例写法，正式业务里还应接安全、作弊、近期活跃等口径
eligible['pass_entry_gate'] = np.where(eligible['active_weeks'] >= 2, 1, 0)

# 轻量校正后的排序
eligible['new_rule_score'] = (
    0.60 * eligible['old_rule_pct'] +
    0.20 * eligible['retention_pct'] +
    0.15 * eligible['efficiency_pct'] +
    0.05 * (1 - eligible['resource_pct_rank'])
)

eligible[['creator_id', 'pass_entry_gate', 'old_rule_score', 'new_rule_score']].head()


## 6. 在同规模 Top N 下做回测

关键原则：
- 不能扩大资源池规模来制造改善；
- 不能只看留存，不看单位激励收入；
- 不能只看离线最优，不看迁移成本。


In [ ]:
pool = eligible[eligible['pass_entry_gate'] == 1].copy()
old_sel = pool.nlargest(4200, 'old_rule_score')
new_sel = pool.nlargest(4200, 'new_rule_score')

backtest = pd.DataFrame({
    '方案': ['原规则', '校正后规则'],
    '30日留存率': [old_sel['retention_30d_rate'].mean(), new_sel['retention_30d_rate'].mean()],
    '单位激励收入': [old_sel['unit_incentive_revenue'].mean(), new_sel['unit_incentive_revenue'].mean()],
    '人均现金激励': [old_sel['total_cash_incentive'].mean(), new_sel['total_cash_incentive'].mean()],
    '高潜命中率': [
        (((old_sel['retention_pct_rank'] >= 0.65) & (old_sel['efficiency_pct_rank'] >= 0.65) & (old_sel['resource_pct_rank'] <= 0.68)).mean()),
        (((new_sel['retention_pct_rank'] >= 0.65) & (new_sel['efficiency_pct_rank'] >= 0.65) & (new_sel['resource_pct_rank'] <= 0.68)).mean())
    ]
})
backtest


### 业务判断

结果解读：

- 人均激励上升，说明资源更集中地投向了质量更高的人；
- 单位激励收入仍然改善，说明成本上升不是无效投入；
- 留存同步改善，说明并非只是短期冲量。


## 7. 迁移成本：名单变化不能太猛


In [ ]:
old_ids = set(old_sel['creator_id'])
new_ids = set(new_sel['creator_id'])

overlap_rate = len(old_ids & new_ids) / len(old_ids)
promoted = len(new_ids - old_ids)
dropped = len(old_ids - new_ids)

pd.DataFrame({
    '指标': ['新旧名单重合率', '新增创作者数', '移出创作者数'],
    '值': [overlap_rate, promoted, dropped]
})


### 业务判断

名单重合率过低会显著抬升迁移成本，因此需要在结果改善与执行成本之间取平衡。


## 8. 稳健性检查

至少应检查两件事：
- 更换 Top N 后，方向是否依旧成立；
- 按内容垂类拆开后，结果是否被单一垂类“抬”出来。


In [ ]:
topn_results = []
for n in [3000, 3600, 4200, 5000, 6000]:
    old_n = pool.nlargest(n, 'old_rule_score')
    new_n = pool.nlargest(n, 'new_rule_score')
    topn_results.append({
        'top_n': n,
        'retention_uplift_ppt': (new_n['retention_30d_rate'].mean() - old_n['retention_30d_rate'].mean()) * 100,
        'unit_incentive_revenue_uplift_pct': (new_n['unit_incentive_revenue'].mean() / old_n['unit_incentive_revenue'].mean() - 1) * 100
    })

pd.DataFrame(topn_results)


## 9. 上线前要准备什么

离线结果再好，也不能直接宣布上线成功。正式推进前，需要把以下内容变成制度化动作：

- creator 级锁定；
- 池间隔离，减少推荐外溢；
- 对降配对象做复核与申诉；
- 每日监控 SRM、现金激励、人均投诉、作弊与安全风险；
- 先看 week 1，再看 week 4，不被短期新鲜感效应误导。


## 10. 最终判断


> 在控制迁移成本和执行风险的前提下，把资源从一部分高曝光低效率人群中挪出，再补向更高质量的稳定供给，是一个值得进入线上灰度的方向。
